In [239]:
import pandas as pd
import numpy as np
df = pd.read_csv("olx_bikes_dataset.csv")
df.head()
df.shape

(7023, 8)

In [240]:
df['KM_driven_per_year'] = df['KM_Driven'] / np.maximum(1,2026-df['Year'])


In [241]:
df.shape

(7023, 9)

In [242]:
df = df[
    (df['KM_driven_per_year'] <= 25000) &
    (df['KM_Driven'] >= 100) &
    (df['KM_Driven'] <= 150000)
].reset_index(drop=True)

In [243]:
df.shape

(5977, 9)

In [244]:
df = df[
    (df['Price'] <= 500000)
].reset_index(drop=True)

In [245]:
df.head()

,Title,Brand,Model,Year,KM_Driven,Price,Area,City,KM_driven_per_year
0,Hero Karizma 2014 Model,Hero,2014 Karizma,2014.0,12800,45000.0,Aurmijivan Society,Valsad,1066.666667
1,Reliable Suzuki Access 125,Suzuki,Access 125,2019.0,35000,52000.0,Mota Varachha,Surat,5000.000000
2,BMW G 310 GS For sale with excellent condition...,BMW,G 310 GS,2025.0,13000,350000.0,Satellite,Ahmedabad,13000.000000
3,Yamaha R15S 2017 bike,Yamaha,R15S,2017.0,40000,55000.0,Vavdi Industrial Area,Rajkot,4444.444444
4,BAJAJ VIKRANT V15 - MODEL 2016,Bajaj,V15,2016.0,35000,29000.0,Paldi,Ahmedabad,3500.000000


In [246]:
df.shape

(5965, 9)

In [247]:
# 1. Count occurrences of each city
city_counts = df['City'].value_counts()

# 2. Identify cities with fewer than 10 occurrences
rare_cities = city_counts[city_counts < 10].index

# 3. Replace rare cities with 'Other'
df['City'] = df['City'].replace(rare_cities, 'Other')



In [248]:
df.drop(['Area','Title'], axis=1, inplace=True)

In [249]:
df.head()

,Brand,Model,Year,KM_Driven,Price,City,KM_driven_per_year
0,Hero,2014 Karizma,2014.0,12800,45000.0,Valsad,1066.666667
1,Suzuki,Access 125,2019.0,35000,52000.0,Surat,5000.000000
2,BMW,G 310 GS,2025.0,13000,350000.0,Ahmedabad,13000.000000
3,Yamaha,R15S,2017.0,40000,55000.0,Rajkot,4444.444444
4,Bajaj,V15,2016.0,35000,29000.0,Ahmedabad,3500.000000


In [250]:
df['Brand'].value_counts()

,count
Brand,
Honda,1488
Hero,986
Bajaj,753
Royal Enfield,551
TVS,517
Suzuki,456
Yamaha,455
Hero Honda,133
OLA,117


In [251]:
df.isnull().sum()

,0
Brand,0
Model,0
Year,0
KM_Driven,0
Price,0
City,0
KM_driven_per_year,0


In [252]:
df['Age'] = 2026 - df['Year']

premium_brands = ['Royal Enfield', 'KTM', 'BMW', 'Kawasaki', 'Harley-Davidson', 'Triumph']
df['Is_Premium_Brand'] = df['Brand'].isin(premium_brands).astype(int)

In [259]:
# Drop severe low-end outliers before cross-validation
df = df[df['Price'] >= 15000]

In [260]:
df.shape

(5852, 9)

In [261]:
y=df['Price']
x=df.drop('Price',axis=1)

In [262]:
x.head()

,Brand,Model,Year,KM_Driven,City,KM_driven_per_year,Age,Is_Premium_Brand
0,Hero,2014 Karizma,2014.0,12800,Valsad,1066.666667,12.0,0
1,Suzuki,Access 125,2019.0,35000,Surat,5000.000000,7.0,0
2,BMW,G 310 GS,2025.0,13000,Ahmedabad,13000.000000,1.0,1
3,Yamaha,R15S,2017.0,40000,Rajkot,4444.444444,9.0,0
4,Bajaj,V15,2016.0,35000,Ahmedabad,3500.000000,10.0,0


In [263]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [264]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer,TransformedTargetRegressor
from sklearn.preprocessing import TargetEncoder,FunctionTransformer
from xgboost import XGBRegressor

target_encode = TargetEncoder(smooth='auto',cv=5,random_state=42,target_type='continuous')

log_transformer = FunctionTransformer(np.log1p,validate=True)

preprocessing = ColumnTransformer(transformers=[
    ('target_encode', target_encode, ['City','Model','Brand']),
    ('log_transform',log_transformer, ['KM_Driven','KM_driven_per_year'])
],remainder='passthrough'
)

xgb_tuned = XGBRegressor(
    learning_rate=0.02,
    n_estimators=1000,
    max_depth=6,             # Dropping depth slightly prevents overfitting on noisy entries
    min_child_weight=3,      # Prevents splits on 1-2 noisy listings
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42
)

# Log-Transform Target Wrapper(only transforms the y i.e is the Price column)
xgb_wrapper = TransformedTargetRegressor(
    regressor=xgb_tuned,
    func=np.log1p,
    inverse_func=np.expm1
)

my_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessing),
    ('model', xgb_wrapper)
])

In [265]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score

kf = KFold(n_splits=5, shuffle=True, random_state=42)

r2 = cross_val_score(my_pipeline, x, y, cv=kf, scoring='r2').mean()

mae = -cross_val_score(my_pipeline, x, y, cv=kf, scoring='neg_mean_absolute_error').mean()

rmse = -cross_val_score(my_pipeline, x, y, cv=kf, scoring='neg_root_mean_squared_error').mean()

# Print Results
print(f"R2 Score : {r2:.4f}")
print(f"MAE      : ₹{mae:,.2f}")
print(f"RMSE     : ₹{rmse:,.2f}")

R2 Score : 0.8211
MAE      : ₹12,908.69
RMSE     : ₹23,793.00


In [266]:
from sklearn.metrics import mean_absolute_percentage_error

# Calculate percentage error across folds
mape = -cross_val_score(my_pipeline, x, y, cv=kf, scoring='neg_mean_absolute_percentage_error').mean()

print(f"MAPE (Percentage Error): {mape * 100:.2f}%")

MAPE (Percentage Error): 21.24%


In [267]:
import joblib

# 1. Train final model on ALL data
my_pipeline.fit(x, y)

# 2. Save pipeline object for production deployment
joblib.dump(my_pipeline, 'xgboost_bike_price_model.pkl')

print("Final model trained on 100% of data and saved successfully!")

Final model trained on 100% of data and saved successfully!
